# Ray casting and sink folds

This notebook is about a *manipulation* of a finished crease pattern rather than a
pipeline that builds one. Three pieces work together:

- [`pleat.ray_casting`](../reference/pleat/ray_casting.md) — send a ray from a point on an
  edge through the pattern, and materialise its trajectory as new creases.
- [`pleat.flat_foldable`](../reference/pleat/flat_foldable.md) — a complete vertex-wise
  local flat-foldability test (Kawasaki, Maekawa, big-little-big, via Hull's crimp recursion).
- [`pleat.sink`](../reference/pleat/sink.md) — the open sink fold: trace a rim, invert
  everything strictly inside it, and infer whether the rim is mountain or valley.

Every cell keeps the knobs worth turning at the top, so the thing to do with this notebook
is to change them and re-run.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from pleat.example_graphs import from_tiles, rosette
from pleat.example_tilesets import platonic
from pleat.flat_foldable import (
    folded_crease_angles,
    is_locally_flat_foldable,
    kawasaki_sum,
    local_assignment_valid,
)
from pleat.overlap import CREASE_ASSIGNMENT, MOUNTAIN, VALLEY, color_creases
from pleat.ray_casting import (
    RAY_CREASE,
    DegenerateRayError,
    add_ray_creases,
    cast_ray,
    halfedge_direction,
    transmit,
)
from pleat.rendering import CREASE_PATTERN_PRESET, multi_show
from pleat.shrink_rotate import assign_this_way_by_bfs, shrink_rotate_pattern
from pleat.sink import InvalidSinkError, open_sink

A few helpers, so that the cells below can name a vertex or a half-edge by *what it is*
rather than by index. `central_vertex()` is deliberately avoided: on a square grid it is an
`argmin` over a four-way tie, so it varies between runs.

In [ ]:
def square_grid(rings=2):
    """A square grid of unit edge length, centred on the origin."""
    G = from_tiles(platonic(n=4), rings=rings)
    G.recompute_lengths_and_angles()
    return G


def vertex_at(G, pos):
    """The vertex at *pos*."""
    return next(v for v in G.vertices if np.allclose(v["pos"], pos, atol=1e-9))


def unit(vector):
    vector = np.asarray(vector, dtype=float)
    return vector / np.linalg.norm(vector)


def outgoing_towards(v, offset):
    """The outgoing half-edge at *v* pointing along *offset*."""
    return next(h for h in v.outgoing_iter() if np.allclose(unit(halfedge_direction(h)), unit(offset), atol=1e-9))


def point_on(h, t):
    """The point at parameter *t* along half-edge *h* -- the start primitive of a cast."""
    return h.orig["pos"] + t * (h.dest["pos"] - h.orig["pos"])


def plot_path(G, path, start=None, ax=None, color="tab:green", title=None):
    """Draw *G* in grey and a cast `RayPath` on top; squares mark hits that landed on a vertex.

    `path.hits` runs from the backward end to the forward end, so the *start* of the cast is
    somewhere in the middle of the list; pass it in to have it marked with a dot.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4.5))
    for h in G.halfedges:
        a, b = h.orig["pos"], h.dest["pos"]
        ax.plot([a[0], b[0]], [a[1], b[1]], color="0.82", lw=1, zorder=0)
    p = np.stack([hit.position for hit in path.hits])
    ax.plot(p[:, 0], p[:, 1], color=color, lw=2)
    if start is not None:
        ax.plot(*start, "o", color=color, ms=7)
    on_vertex = [hit.position for hit in path.hits if hit.vertex is not None]
    if on_vertex:
        q = np.stack(on_vertex)
        ax.plot(q[:, 0], q[:, 1], "s", color="tab:orange", ms=7, ls="none")
    ax.set_aspect("equal")
    ax.invert_yaxis()  # match the orientation of pleat's own cairo renderer, used from section 4 on
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax

## 1. Casting a ray

[`cast_ray`](../reference/pleat/ray_casting.md#pleat.ray_casting.cast_ray) starts from a
point on an edge, given as a half-edge and a parameter `t ∈ [0, 1]`, plus a direction. At
every crease it meets, the direction *transmits*:

$$ d' = d - 2\,(d \cdot \hat u)\,\hat u $$

with $\hat u$ the unit direction of the crease. This keeps the component that crosses the
crease and flips the component along it. The ray below starts in the middle of a vertical
edge and heads east at a shallow angle; each vertical crease flips its $y$-component, so it
zig-zags across the sheet.

In [ ]:
START_POS = [-0.5, 0.5]  # which grid vertex the start edge hangs off
DIRECTION = np.array([1.0, 0.4])

G = square_grid()
start = outgoing_towards(vertex_at(G, START_POS), [0.0, 1.0])
path = cast_ray(G, start, 0.5, DIRECTION)

print("ends:", path.ends, " closed:", path.closed, " hits:", len(path.hits))
plot_path(G, path, start=point_on(start, 0.5), title="one ray; the dot is where it was cast from")
plt.show()

Transmission is **not** a mirror reflection. Mirroring across the crease line would flip the
crossing component instead, sending the ray straight back into the face it came from:

In [ ]:
d = unit([1.0, 0.0])
u = unit([1.0, 1.0])  # a 45° crease
print("transmitted:", np.round(transmit(d, u), 6))  # a 90° turn
print("mirrored:   ", np.round(2 * np.dot(d, u) * u - d, 6))  # back the way it came

## 2. Hitting a vertex head-on

The subtle case is a ray aimed exactly at a vertex. It is resolved by treating the ray as
passing an infinitesimal distance $\varepsilon$ to one side and asking which creases that
offset line meets — a question that turns out to be purely angular, because $\varepsilon$
cancels exactly. The walk is a fan around the vertex driven by the alternating prefix sum of
the sector angles, so it can transmit through *several* creases at one vertex.

`side` picks which side the ray passes on, and the two answers are genuine mirror images of
each other. Below, the same ray is aimed dead-centre at the vertex at `(-0.5, 0.5)`: passing
on the left it turns and closes a small loop, passing on the right it carries straight on to
the border. Orange squares mark hits that landed on a vertex.

In [ ]:
AIM_AT = [-0.5, 0.5]
FROM_VERTEX = [-1.5, -0.5]  # start edge runs north from here

G = square_grid()
start = outgoing_towards(vertex_at(G, FROM_VERTEX), [0.0, 1.0])
direction = np.asarray(AIM_AT) - (start.orig["pos"] + start.dest["pos"]) / 2

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, side in zip(axes, ["left", "right"]):
    p = cast_ray(G, start, 0.5, direction, side=side, both_ways=False)
    plot_path(G, p, start=point_on(start, 0.5), ax=ax, title=f"side={side!r}, ends={p.ends}")
plt.show()

## 3. Both ways, and starting at a node

`both_ways=True` (the default) completes the trajectory when the forward ray runs off the
paper: a second ray is cast from the same start point and its hits are prepended, reversed
and re-oriented, so `hits` reads as one curve from the backward end to the forward end. The
ray in section 1 does exactly this — its `ends` are `('border', 'border')`.

The backward heading is *not* simply `-d`. The start point lies **on** the start edge, so the
full trajectory crosses that edge there: it arrives along `transmit(d, E)` and departs along
`d`, and retracing the arrival means leaving along `-transmit(d, E)`. Anything else puts a
kink in the curve at the one place it is guaranteed to cross an existing crease.

In [ ]:
G = square_grid()
start = outgoing_towards(vertex_at(G, [-0.5, 0.5]), [0.0, 1.0])
path = cast_ray(G, start, 0.5, np.array([1.0, 0.4]))

for i, hit in enumerate(path.hits):
    print(
        f"{i}: pos={np.round(hit.position, 3)}  out={np.round(hit.direction_out, 3)}"
        f"  on_vertex={hit.vertex is not None}  crossed={len(hit.halfedges)}"
    )

One wrinkle worth knowing when walking `hits`: on the shared start hit (index 2 above)
`direction_in` is the *departure* heading rather than the arrival heading, because both
passes emit that hit and the forward pass's copy is the one kept. Every other hit has
`direction_in` equal to the previous hit's `direction_out`.

`t = 0` and `t = 1` mean the half-edge's endpoints, so **starting at a node is the same entry
point**, not a separate one. Which face the ray sets off into is then decided by the *sector*
holding the direction, not by which side of the start edge it points to — so the cast does
not depend on which of the incident half-edges was used to name the node. All four agree:

In [ ]:
NODE = [-0.5, 0.5]
DIRECTION = np.array([1.0, 0.4])

G = square_grid()
v = vertex_at(G, NODE)
reference = None
for offset in ([0, 1], [1, 0], [0, -1], [-1, 0]):
    p = cast_ray(G, outgoing_towards(v, offset), 0.0, DIRECTION)
    points = np.stack([hit.position for hit in p.hits])
    reference = points if reference is None else reference
    print(f"named by the half-edge towards {offset}: ends={p.ends}, same path={np.allclose(points, reference)}")

## 4. Adding the creases

[`add_ray_creases`](../reference/pleat/ray_casting.md#pleat.ray_casting.add_ray_creases)
casts the ray to completion and *then* materialises it — the ray must be traced on the
original pattern, or its own new creases would deflect it. It returns the new half-edges in
traversal order together with the `RayPath`; every new half-edge is tagged with
`RAY_CREASE`.

In [ ]:
G = square_grid()
before = G.copy()
start = outgoing_towards(vertex_at(G, [-0.5, 0.5]), [0.0, 1.0])

rim, path = add_ray_creases(G, start, 0.5, np.array([1.0, 0.4]))

print(f"{len(rim)} new half-edges, all tagged: {all(h[RAY_CREASE] for h in rim)}")
print("continuous:", all(a.dest is b.orig for a, b in zip(rim, rim[1:])))
G.check_consistency()

for h in rim:  # colour the ray so it stands out against the grid
    h["color_key"] = h.rev["color_key"] = (0.1, 0.6, 0.2)
multi_show([before, G], titles=["before", "after add_ray_creases"], line_width=0.03, **CREASE_PATTERN_PRESET)

## 5. The open sink

An open sink pushes a folded point through the model. In the crease pattern that is: trace a
rim with a ray, invert every crease **strictly inside** it, and give the rim itself a single
uniform assignment. Which one is not a free choice —
[`open_sink`](../reference/pleat/sink.md#pleat.sink.open_sink) sets the whole rim to
`MOUNTAIN`, tests every interior rim node for local flat-foldability, and if that fails flips
the whole rim to `VALLEY` and tests again.

The fixture below is the classic case: a square grid with the four diagonals added at one
vertex, giving a degree-8 vertex whose 45° sectors turn a passing ray by 90° each time, so a
ray leaving the midpoint of the north crease traces a closed square rim around it.

In [ ]:
def diagonal_grid(diagonals=MOUNTAIN, centre=(0.5, 0.5)):
    """A square grid with all four diagonals added at *centre*, everything mountain but those."""
    G = square_grid()
    v = vertex_at(G, centre)
    for towards, corner in (([0, 1], [-1, 1]), ([-1, 0], [-1, -1]), ([0, -1], [1, -1]), ([1, 0], [1, 1])):
        face = outgoing_towards(v, towards).face
        far = next(
            w for w in face.vertex_iter() if w is not v and np.allclose(unit(w["pos"] - v["pos"]), unit(corner))
        )
        G.subdivide_face(face, v, far)
    G.recompute_lengths_and_angles()
    for h in G.halfedges:
        h[CREASE_ASSIGNMENT] = MOUNTAIN
    for corner in ([-1, 1], [-1, -1], [1, -1], [1, 1]):
        h = outgoing_towards(vertex_at(G, centre), corner)
        h[CREASE_ASSIGNMENT] = h.rev[CREASE_ASSIGNMENT] = diagonals
    return G


DIAGONALS = VALLEY  # try MOUNTAIN as well; the rim follows

G = diagonal_grid(DIAGONALS)
before = G.copy()
start = outgoing_towards(vertex_at(G, [0.5, 0.5]), [0.0, 1.0])

rim = open_sink(G, start, 0.5, np.array([-1.0, 0.0]))

print("rim length:", len(rim), " closed:", rim[-1].dest is rim[0].orig)
print("rim assignment:", {"mountain" if h[CREASE_ASSIGNMENT] == MOUNTAIN else "valley" for h in rim})
color_creases(before)
color_creases(G)
multi_show(
    [before, G],
    titles=["before the sink", "after the sink"],
    line_width=0.03,
    **CREASE_PATTERN_PRESET,
)

Mountain is red, valley blue. Inside the rim the eight radial half-creases have swapped
colour and nothing outside it has moved; the rim itself came out uniformly valley, forced by
big-little-big at the four corner nodes, where the sectors are 135°/45°/45°/135° and the
inversion has made the two halves of the diagonal disagree. Set `DIAGONALS = MOUNTAIN` above
and the whole rim flips with them.

The verdict at each rim node comes from
[`local_assignment_valid`](../reference/pleat/flat_foldable.md#pleat.flat_foldable.local_assignment_valid),
which returns `(valid, margin)`. There is no meaningful scalar residual for M/V validity —
it is a discrete predicate — so `margin` reports the robustness instead: the smallest gap
between distinct folded crease positions. A small margin means the vertex is genuinely
ambiguous, not that it is nearly wrong.

Those folded positions are
[`folded_crease_angles`](../reference/pleat/flat_foldable.md#pleat.flat_foldable.folded_crease_angles),
the alternating prefix sum $\psi_k = \psi_{k-1} + (-1)^k a_k$ of the sector angles — the same
walk the vertex fan of section 2 performs. Its last entry *is* the Kawasaki sum: Kawasaki is
the statement that this cycle closes, not an independent condition.

In [ ]:
node = rim[0].dest
print("degree:            ", node.order())
print("psi:               ", np.round(folded_crease_angles(node), 6))
print("psi[-1] == kawasaki:", np.isclose(folded_crease_angles(node)[-1], kawasaki_sum(node)))
print("local_assignment_valid:", local_assignment_valid(node))

### End to end on a flat-foldable pattern

The grid above is all-mountain, so it satisfies Maekawa nowhere and only its rim nodes can be
checked. To make the full claim — *a successful sink turns a valid crease pattern into a
valid crease pattern* — start from a shrink-rotate tessellation, which is genuinely locally
flat-foldable, and check the whole graph with
[`is_locally_flat_foldable`](../reference/pleat/flat_foldable.md#pleat.flat_foldable.is_locally_flat_foldable)
before and after.

In [ ]:
SINK_ANGLE = 305  # degrees; 65, 245 also give closed rims on this pattern

tiling = from_tiles(platonic(n=6), rings=1)
assign_this_way_by_bfs(tiling, tiling.central_face())
CP = shrink_rotate_pattern(tiling, simplify_boundary=True, alpha=np.pi / 5, factor=0.5)
CP.recompute_lengths_and_angles()
before = CP.copy()

# a vertex picked by proximity to a point only one vertex is anywhere near, so the choice is
# stable between runs, unlike an argmin over near-ties
v = min(CP.vertices, key=lambda w: float(np.linalg.norm(np.asarray(w["pos"]) - [0.5, 0.05])))
start = min(v.outgoing_iter(), key=lambda h: float(np.linalg.norm(np.asarray(h.dest["pos"]) - [0.2, 0.46])))
direction = np.array([np.cos(np.radians(SINK_ANGLE)), np.sin(np.radians(SINK_ANGLE))])

def mv_counts(graph):
    return {
        "mountain": sum(h.get(CREASE_ASSIGNMENT) == MOUNTAIN for h in graph.halfedges),
        "valley": sum(h.get(CREASE_ASSIGNMENT) == VALLEY for h in graph.halfedges),
    }


print("flat-foldable before:", is_locally_flat_foldable(CP)[0], mv_counts(CP))
rim = open_sink(CP, start, 0.5, direction)
ok, violations = is_locally_flat_foldable(CP)
print("flat-foldable after: ", ok, mv_counts(CP), "" if ok else {tuple(map(float, w["pos"])): m for w, m in violations.items()})
print("rim:", len(rim), "edges, uniformly", "mountain" if rim[0][CREASE_ASSIGNMENT] == MOUNTAIN else "valley")
CP.check_consistency()

color_creases(before)
color_creases(CP)
multi_show(
    [before, CP],
    titles=["shrink-rotate CP", f"sunk (ray at {SINK_ANGLE}°)"],
    line_width=0.03,
    **CREASE_PATTERN_PRESET,
)

The rim here is the small quadrilateral just right of centre — a sink of one twist corner
rather than of a whole rosette, which is what closes on this pattern. What matters is the
printout: the pattern was locally flat-foldable before and still is after, the
mountain/valley tally has shifted, and the rim came out uniform.

## 6. When it refuses

**A self-crossing ray.** An origami sink rim is a simple curve. If the ray crosses its own
earlier path, the earlier chord has already split the face, so the later segment has one
endpoint on each side of it and cannot be laid; silently skipping it would leave a gap in the
rim and the interior flood fill would leak through it. So it raises — and, because the
crossing is detected on the finished trajectory *before* anything is materialised, `G` is
left untouched.

In [ ]:
G = rosette(7)
G.recompute_lengths_and_angles()
spoke = min(G.central_vertex().outgoing_iter(), key=lambda h: float(np.linalg.norm(unit(halfedge_direction(h)) - unit([-0.6, -0.8]))))
edges_before = len(G.halfedges)

try:
    add_ray_creases(G, spoke, 0.5, np.array([1.0, 0.0]))
except DegenerateRayError as error:
    print(f"DegenerateRayError: {error}")

print("half-edges unchanged:", len(G.halfedges) == edges_before)
G.check_consistency()

**A sink that cannot be assigned.** Make two of the four diagonals valley and two mountain,
and the four corner nodes disagree about which way the rim must go. Neither uniform
assignment works, so `open_sink` raises. Note that unlike the case above it has *already*
modified `G` — the rim is materialised and the interior inverted before the assignment is
attempted — so work on a copy if you want to keep the original.

In [ ]:
G = diagonal_grid(MOUNTAIN)
v = vertex_at(G, [0.5, 0.5])
for corner in ([-1, 1], [-1, -1]):  # two of four, so the corners cannot agree
    h = outgoing_towards(v, corner)
    h[CREASE_ASSIGNMENT] = h.rev[CREASE_ASSIGNMENT] = VALLEY

try:
    open_sink(G, outgoing_towards(v, [0.0, 1.0]), 0.5, np.array([-1.0, 0.0]))
except InvalidSinkError as error:
    print(f"InvalidSinkError: {error}")

**A pattern with no crease assignment at all.** `strict=True` refuses; `strict=False` logs
and carries on. `strict` is purely a crease-assignment switch — the geometry it produces is
identical either way, which makes `open_sink` usable as a pure geometric operation on a
pattern that has not been assigned yet.

In [ ]:
def bare_grid():
    """The same fixture with every crease assignment stripped off."""
    G = diagonal_grid(MOUNTAIN)
    for h in G.halfedges:
        del h.attributes[CREASE_ASSIGNMENT]
    return G, outgoing_towards(vertex_at(G, [0.5, 0.5]), [0.0, 1.0])


G, start = bare_grid()
try:
    open_sink(G, start, 0.5, np.array([-1.0, 0.0]))
except InvalidSinkError as error:
    print(f"strict=True  -> InvalidSinkError: {error}")

G, start = bare_grid()
rim = open_sink(G, start, 0.5, np.array([-1.0, 0.0]), strict=False)
G.check_consistency()
print("strict=False -> rim of", len(rim), "edges; vertices:", len(G.vertices))

A last failure mode worth knowing about: a cast reports how each end stopped in
`RayPath.ends`, and only `"closed"` and `"border"` mean an end was traced to completion.
Test for failure with `ends[i] not in ("closed", "border")` rather than looking for
`"max_steps"` — `"stalled"` and `"start"` are failures too, and more reasons may be added.
`open_sink` applies exactly that rule, and additionally refuses `both_ways=False`: a one-way
rim has a loose end in mid-sheet, does not separate the paper, and would invert the entire
model — which no local flat-foldability check could ever see, since a global mountain/valley
flip preserves every local condition.